# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bilalahmed251/-ML-Search-Discovery/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# I will rank pages for human review using a transparent action score based on visibility, low CTR, weaker average position, content age,
# and time since the last update. Each ranked page will receive reason codes so that a reviewer can understand why it was prioritized.
# The ranking is directional decision-support, not an automatic instruction to refresh every page.


In [16]:
import os
import pandas as pd
import numpy as np

repo = "/content/ML-Search-Discovery"

if not os.path.exists(repo):
    !git clone -q --depth 1 https://github.com/bilalahmed251/-ML-Search-Discovery.git {repo}

df_playbook = pd.read_csv(
    f"{repo}/data/raw/content_refresh_anonymized.csv"
 ).copy()

required_fields = [
    "content_id",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
]

missing_fields = [
    field for field in required_fields
    if field not in df_playbook.columns
]

print("Missing required fields:", missing_fields)

queue = df_playbook[required_fields].copy()

for field in required_fields[1:]:
    queue[field] = pd.to_numeric(
        queue[field],
        errors="coerce"
    )

queue = queue.dropna().copy()

# Percentile-based signals
queue["visibility_score"] = (
    queue["impressions_90d"].rank(pct=True)
)

queue["low_ctr_score"] = (
    1 - queue["ctr"].rank(pct=True)
)

queue["weak_position_score"] = (
    queue["avg_position"].rank(pct=True)
)

queue["old_content_score"] = (
    queue["content_age_days"].rank(pct=True)
)

queue["stale_update_score"] = (
    queue["days_since_last_update"].rank(pct=True)
)

# Transparent action-priority score
queue["action_score"] = (
    0.30 * queue["visibility_score"] +
    0.20 * queue["low_ctr_score"] +
    0.20 * queue["weak_position_score"] +
    0.15 * queue["old_content_score"] +
    0.15 * queue["stale_update_score"]
)

# Reason codes
def make_reason_codes(row):
    reasons = []

    if row["visibility_score"] >= 0.75:
        reasons.append("HIGH_VISIBILITY")

    if row["low_ctr_score"] >= 0.75:
        reasons.append("LOW_CTR")

    if row["weak_position_score"] >= 0.75:
        reasons.append("WEAK_POSITION")

    if row["old_content_score"] >= 0.75:
        reasons.append("OLD_CONTENT")

    if row["stale_update_score"] >= 0.75:
        reasons.append("STALE_UPDATE")

    return ", ".join(reasons) if reasons else "REVIEW_SIGNALS"

queue["reason_codes"] = queue.apply(
    make_reason_codes,
    axis=1
)

queue = queue.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

print("Ranked pages:", len(queue))
display(queue.head(10))


Missing required fields: []
Ranked pages: 30000


,content_id,impressions_90d,ctr,avg_position,content_age_days,days_since_last_update,visibility_score,low_ctr_score,weak_position_score,old_content_score,stale_update_score,action_score,reason_codes
0,content_fb4bf6555c79,84093,0.00,45.6,299,104,0.992367,0.779783,0.942083,0.619517,0.843200,0.861491,"HIGH_VISIBILITY, LOW_CTR, WEAK_POSITION, STALE..."
1,content_8d0a8cbf9d1e,5090,0.00,47.7,537,104,0.797333,0.779783,0.948900,0.987917,0.843200,0.859604,"HIGH_VISIBILITY, LOW_CTR, WEAK_POSITION, OLD_C..."
2,content_05133844bff4,14803,0.00,54.4,482,22,0.917433,0.779783,0.967083,0.934717,0.588283,0.853053,"HIGH_VISIBILITY, LOW_CTR, WEAK_POSITION, OLD_C..."
3,content_75175d878762,25748,0.00,48.5,299,104,0.955700,0.779783,0.950967,0.619517,0.843200,0.852267,"HIGH_VISIBILITY, LOW_CTR, WEAK_POSITION, STALE..."
4,content_d376881f5bd0,5301,0.00,54.8,445,104,0.803583,0.779783,0.967867,0.866900,0.843200,0.847120,"HIGH_VISIBILITY, LOW_CTR, WEAK_POSITION, OLD_C..."
5,content_9ecd60ff3bca,4543,0.00,71.3,445,104,0.782350,0.779783,0.991200,0.866900,0.843200,0.845417,"HIGH_VISIBILITY, LOW_CTR, WEAK_POSITION, OLD_C..."
6,content_bc18d49d8f6b,32491,0.00,36.2,287,104,0.968133,0.779783,0.896800,0.594033,0.843200,0.841342,"HIGH_VISIBILITY, LOW_CTR, WEAK_POSITION, STALE..."
7,content_095661034f9b,23513,0.00,39.4,287,104,0.951233,0.779783,0.914617,0.594033,0.843200,0.839835,"HIGH_VISIBILITY, LOW_CTR, WEAK_POSITION, STALE..."
8,content_5292478e83f6,4764,0.00,47.1,445,104,0.789033,0.779783,0.947150,0.866900,0.843200,0.838612,"HIGH_VISIBILITY, LOW_CTR, WEAK_POSITION, OLD_C..."
9,content_62abc4bd66be,31364,0.09,68.5,445,104,0.966083,0.472150,0.988333,0.866900,0.843200,0.838437,"HIGH_VISIBILITY, WEAK_POSITION, OLD_CONTENT, S..."


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# The action queue is intended for SEO editors, content strategists, and analysts who need to prioritize pages for human review.
# It can support decisions about which pages may need a refresh, stronger search intent alignment, clearer titles and descriptions, or improved content depth.
# It does not automatically publish edits, guarantee traffic growth, prove causation, or replace editorial judgment.
# The queue is a directional decision-support tool based on observed page-level signals.


In [18]:
intended_use = {
    "primary_users": [
        "SEO editors",
        "Content strategists",
        "Analytics reviewers"
    ],
    "valid_uses": [
        "Prioritize pages for human review",
        "Identify possible low-CTR opportunities",
        "Identify pages with weak search position",
        "Identify older or stale pages"
    ],
    "not_valid_uses": [
        "Automatically publish content changes",
        "Guarantee traffic growth",
        "Claim that refresh caused improvement",
        "Replace human editorial review"
    ]
}

for category, items in intended_use.items():
    print(f"\n{category}:")
    for item in items:
        print(f"- {item}")



primary_users:
- SEO editors
- Content strategists
- Analytics reviewers

valid_uses:
- Prioritize pages for human review
- Identify possible low-CTR opportunities
- Identify pages with weak search position
- Identify older or stale pages

not_valid_uses:
- Automatically publish content changes
- Guarantee traffic growth
- Claim that refresh caused improvement
- Replace human editorial review


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Before acting on a recommendation, a human reviewer must check the page's search intent, current title and description, content quality, accuracy, originality,
# seasonality, recent updates, and tracking reliability. A reviewer should also check whether the page has business or legal importance that is not represented in the dataset.
# The system must never automatically publish edits, delete content, change canonical tags, redirect URLs, or make claims about causation without human approval.


In [20]:
human_review_checklist = [
    "Confirm the page search intent",
    "Review the current title and description",
    "Check content accuracy and completeness",
    "Check originality and duplication risk",
    "Check seasonality and recent changes",
    "Verify analytics and tracking quality",
    "Check business or legal importance",
]

no_go_actions = [
    "Automatically publish edits",
    "Delete content automatically",
    "Change canonical tags automatically",
    "Redirect URLs automatically",
    "Claim that a refresh caused improvement",
]

print("Human review checklist:")
for item in human_review_checklist:
    print("-", item)

print("\nNo-go actions:")
for item in no_go_actions:
    print("-", item)


Human review checklist:
- Confirm the page search intent
- Review the current title and description
- Check content accuracy and completeness
- Check originality and duplication risk
- Check seasonality and recent changes
- Verify analytics and tracking quality
- Check business or legal importance

No-go actions:
- Automatically publish edits
- Delete content automatically
- Change canonical tags automatically
- Redirect URLs automatically
- Claim that a refresh caused improvement


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# The recommendations should be reviewed when the search-data window becomes old, when traffic or ranking distributions change,
# when the declining-page rate shifts materially, or when editors observe many poor recommendations.
# The model should be revalidated after major changes to search behavior, tracking, content templates, or feature definitions.
# A time-based evaluation and a prospective refresh experiment should be completed before treating the queue as reliable for ongoing use.


In [22]:
monitoring_triggers = {
    "data_freshness": "Review when the search-data window becomes outdated.",
    "distribution_shift": "Review when traffic, CTR, or position distributions change materially.",
    "label_shift": "Review when the declining-page rate changes materially.",
    "human_feedback": "Review when editors report many poor recommendations.",
    "system_change": "Revalidate after major changes to tracking, templates, or search behavior.",
    "scheduled_review": "Recheck the model on a regular recurring schedule.",
}

for trigger, explanation in monitoring_triggers.items():
    print(f"{trigger}: {explanation}")


data_freshness: Review when the search-data window becomes outdated.
distribution_shift: Review when traffic, CTR, or position distributions change materially.
label_shift: Review when the declining-page rate changes materially.
human_feedback: Review when editors report many poor recommendations.
system_change: Revalidate after major changes to tracking, templates, or search behavior.
scheduled_review: Recheck the model on a regular recurring schedule.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [23]:
# This cell is for CODE (numbers, a query, a check).
# I will export the ranked action queue to the work/outputs folder so it can be reused in the capstone and paper.
# The export contains the page identifier, observed signals, action score, and reason codes.
# It is a review queue for decision-support and does not represent completed content changes or guaranteed outcomes.


In [24]:
output_dir = f"{repo}/work/outputs"
os.makedirs(output_dir, exist_ok=True)

export_columns = [
    "content_id",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "visibility_score",
    "low_ctr_score",
    "weak_position_score",
    "old_content_score",
    "stale_update_score",
    "action_score",
    "reason_codes",
]

export_columns = [
    col for col in export_columns
    if col in queue.columns
]

action_queue_path = (
    f"{output_dir}/content_action_playbook.csv"
)

queue[export_columns].to_csv(
    action_queue_path,
    index=False
)

print("Saved:", action_queue_path)
print("Exported rows:", len(queue))
print("Exported columns:", len(export_columns))

display(
    pd.read_csv(action_queue_path).head(10)
)


Saved: /content/ML-Search-Discovery/work/outputs/content_action_playbook.csv
Exported rows: 30000
Exported columns: 13


,content_id,impressions_90d,ctr,avg_position,content_age_days,days_since_last_update,visibility_score,low_ctr_score,weak_position_score,old_content_score,stale_update_score,action_score,reason_codes
0,content_fb4bf6555c79,84093,0.00,45.6,299,104,0.992367,0.779783,0.942083,0.619517,0.843200,0.861491,"HIGH_VISIBILITY, LOW_CTR, WEAK_POSITION, STALE..."
1,content_8d0a8cbf9d1e,5090,0.00,47.7,537,104,0.797333,0.779783,0.948900,0.987917,0.843200,0.859604,"HIGH_VISIBILITY, LOW_CTR, WEAK_POSITION, OLD_C..."
2,content_05133844bff4,14803,0.00,54.4,482,22,0.917433,0.779783,0.967083,0.934717,0.588283,0.853053,"HIGH_VISIBILITY, LOW_CTR, WEAK_POSITION, OLD_C..."
3,content_75175d878762,25748,0.00,48.5,299,104,0.955700,0.779783,0.950967,0.619517,0.843200,0.852267,"HIGH_VISIBILITY, LOW_CTR, WEAK_POSITION, STALE..."
4,content_d376881f5bd0,5301,0.00,54.8,445,104,0.803583,0.779783,0.967867,0.866900,0.843200,0.847120,"HIGH_VISIBILITY, LOW_CTR, WEAK_POSITION, OLD_C..."
5,content_9ecd60ff3bca,4543,0.00,71.3,445,104,0.782350,0.779783,0.991200,0.866900,0.843200,0.845417,"HIGH_VISIBILITY, LOW_CTR, WEAK_POSITION, OLD_C..."
6,content_bc18d49d8f6b,32491,0.00,36.2,287,104,0.968133,0.779783,0.896800,0.594033,0.843200,0.841342,"HIGH_VISIBILITY, LOW_CTR, WEAK_POSITION, STALE..."
7,content_095661034f9b,23513,0.00,39.4,287,104,0.951233,0.779783,0.914617,0.594033,0.843200,0.839835,"HIGH_VISIBILITY, LOW_CTR, WEAK_POSITION, STALE..."
8,content_5292478e83f6,4764,0.00,47.1,445,104,0.789033,0.779783,0.947150,0.866900,0.843200,0.838612,"HIGH_VISIBILITY, LOW_CTR, WEAK_POSITION, OLD_C..."
9,content_62abc4bd66be,31364,0.09,68.5,445,104,0.966083,0.472150,0.988333,0.866900,0.843200,0.838437,"HIGH_VISIBILITY, WEAK_POSITION, OLD_CONTENT, S..."


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.